**Key Deliverables**
- TextCleaner class with 6+ normalization methods ✅
- Abbreviation dictionary with 30+ mappings ✅
- Test suite with 40+ test cases covering edge cases ✅
- Cleaned dataset generated from Week 1 sample ✅
- Before/after examples showing improvements ✅
- NEW: Data profiling report showing what needs cleaning (null rates, HTML 
presence, common abbreviations) ✅

In [1]:
import re
import pandas as pd
from collections import Counter
import re
import unicodedata
import nltk

In [2]:
nltk.download('words')

[nltk_data] Downloading package words to
[nltk_data]     C:\Users\mayab\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!


True

In [3]:
df = pd.read_csv("../data/processed/listing_sample.csv")

In [4]:
df.head()

,L_ListingID,L_Address,L_City,beds,baths,price,remarks
0,1169503734,133 Crystal Street,Taft,3.0,2.0,235000,This unique property offers two homes on one l...
1,1154220129,15664 Kadota,Victorville,4.0,4.0,459000,Beautiful Two-Story Home in Victorville – Spac...
2,1159921951,949 S Manhattan Place 203,Los Angeles,3.0,2.0,689000,Presenting this exquisite second-floor corner ...
3,1170333192,1872 Seascape Boulevard,Aptos,3.0,3.0,1195000,Thoughtfully renovated coastal retreat tucked ...
4,1152463169,2085 Westhampton Drive,Arroyo Grande,3.0,2.0,1050000,Welcome to 2085 Westhampton Drive in Arroyo Gr...


In [5]:
df.dtypes

L_ListingID      int64
L_Address          str
L_City             str
beds           float64
baths          float64
price            int64
remarks            str
dtype: object

In [6]:
df.columns

Index(['L_ListingID', 'L_Address', 'L_City', 'beds', 'baths', 'price',
       'remarks'],
      dtype='str')

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   L_ListingID  1000 non-null   int64  
 1   L_Address    999 non-null    str    
 2   L_City       995 non-null    str    
 3   beds         997 non-null    float64
 4   baths        1000 non-null   float64
 5   price        1000 non-null   int64  
 6   remarks      1000 non-null   str    
dtypes: float64(2), int64(2), str(3)
memory usage: 54.8 KB


In [8]:
df.isnull().sum()

L_ListingID    0
L_Address      1
L_City         5
beds           3
baths          0
price          0
remarks        0
dtype: int64

In [9]:
df['L_Address'].isnull().mean()

np.float64(0.001)

In [10]:
df['L_City'].isnull().mean()

np.float64(0.005)

In [11]:
df['beds'].isnull().mean()

np.float64(0.003)

In [12]:
df['L_City'].value_counts()

L_City
Los Angeles     63
San Diego       31
San Jose        27
Irvine          17
Palm Desert     14
                ..
Long Barn        1
Oxnard           1
Fresno           1
North Tustin     1
Encino           1
Name: count, Length: 341, dtype: int64

In [13]:
df['remarks'].str.len().describe()

count    1000.000000
mean     1290.093000
std       578.506257
min        74.000000
25%       883.750000
50%      1221.500000
75%      1613.250000
max      3974.000000
Name: remarks, dtype: float64

In [14]:
df["remarks"].str.contains(r"<[^>]+>", regex=True, na=False).sum()

np.int64(0)

In [119]:
class TextCleaner: 
    def __init__(self): 
        # Abbreviation map
        self.abbrev_map = { 
        'bd': 'bedroom', 'br': 'bedroom', 'ba': 'bathroom', 'sqft': 'square feet', 
        'w/': 'with', 'w/o': 'without', 'mbr': 'master bedroom', "gar": "garage",
        'adu': 'accessory dwelling unit', 'hoa': 'Homeowners Association',
        'sq ft': 'square feet', 'sq': 'square', 'ft': 'feet', 'condo': 'condominium',
        'rv': 'recreational vehicle', 'ev': 'electric vehicle', 'hvac': 'heating, ventilation, and air conditioning',
        'ac': 'air conditioning', 'sf': 'square feet', 'va': 'Veterans Affairs',
        'fha': 'Federal Housing Administration', 'blvd': 'boulevard', 'bbq': 'barbecue', 'tv': 'television',
        "hwy": "highway", "dr": "drive", "rd": "road", "metro": "metropolitan", "bbqs": "barbecues",
        "apn": "Assessors Parcel Number", "combo": "combination", "lvp": "luxury vinyl plank", 
        "etc": "et cetera", "ct": "court", "mt": "mount", "uc": "university of california",
        "jr": "junior", "pga": "professional golfers' association", "rvs": "recreational vehicles", 
        "fwy": "freeway", "eco": "ecology", "thru": "through", "ucla": "University of California, Los Angeles",
        "decor": "decoration", "oc": "orange county", "socal": "southern california", "dtla": "downtown Los Angeles", 
        "appx": "approximately", "str": "short-term rental", "cvs": "consumer value stores",
        "pusd": "Unified School District", "tlc": "tender loving cleaner", "sofi": "social finance", 
        "bdrm": "bedroom", "sdsu": "San Diego State University", "jadu": "junior accessory dwelling unit",
        "ss": "stainless steel", "spc": "stone plastic composite", "adt": "american district telegraph",
        "ln": "lane", "bdrms": "bedrooms","flrs": "floors", "fam": "family"
        }
        self.words_to_ignore = {
            "views", "has", "areas", "parks", "doors", "newer", "ll", "feet", "trees", "hills", "flows", 
            "opens", "baths", "multi", "homes", "adds", "dryer", "pools", "cul", "shops", "rooms", "sinks", 
            "walls", "feels", "los", "acres", "steps", "makes", "owned", "sits", "paid", "units", "fans", "years", 
            "dues", "meets", "costs", "ins", "lakes", "mini", "spas", "sets", "meals", "plans", "lines", 
            "spans", "oaks", "mello", "roos", "gates", "decks", "toys", "paths", "fills", "tesla", 
            "taxes", "beds", "stars", "fees", "foods", "cafes", "ve", "pets", "epoxy", "viejo", 
            "beams", "clubs", "gives", "cared", "uses", "shows", "miele", "boxes", "yards", "lives",
            "jolla", "cars", "runs", "cruz", "tiles", "spots", "palms", "loved", "wells", "ovens", "lg", "rey", 
            "paseo", "yorba", "tons", "mahal", "pits", "poway", "looks", "means", "keeps", "vibe", "chefs", 
            "users", "arts", "pre", "com", "ii", "indio", "italy", "petco", "roads", "tones", "woods", "chula", 
            "mateo", "sheds", "zones", "pairs", "mills", "takes", "palos", "cafe", "alta", "famed", "bars", "boats", 
            "farms", "bi", "amp", "brea", "isn", "doesn", "hubs", "paved", "sales", "dacor", "studs", "banks", "sonos", 
            "norco", "laws", "terms", "ones", "app", "pipes", "walks", "edges", "fits", "joes", "co", "hours", "bones", 
            "heads", "lanai", "pours", "pex", "ups", "terra", "helps", "falls", "rates", "aptos", "azul", "hosts", 
            "joses", "perks", "faces", "tubs", "bikes", "fewer", "adams", "youre", "youll", "bays", "bills", "roots",
            "paso", "backs", "dates", "using", "lawns", "iid", "lagos"
        }
        self.stop_words = {
            "and", "the", "a", "with", "to", "in", "of", "for", "this", "is", "an", 
            "or", "on", "from", "that"
        }
        self.proper_case = {
            "hoa",
            "fha",
            "ucla",
            "pusd",
            "apn",
            "sdsu",
            "va"
        }
    def clean_text(self, text):
        text = self.normalize_html(text) 
        text = self.normalize_unicode(text) 
        text = self.normalize_smart_quotes(text)
        text = self.normalize_prices(text) 
        text = self.normalize_measurements(text) 
        text = self.expand_abbreviations(text) 
        text = self.normalize_whitespace(text)
        return text.strip() 
    def normalize_unicode(self, text):
        # Standardizes Unicode forms and removes invisible Unicode characters."""
        if not isinstance(text, str):
            return ""
        # Fix common Windows-1252 mojibake
        text = (
            text.replace("â€™", "'")
                .replace("â€œ", '"')
                .replace("â€\x9d", '"')
                .replace("Â", "")
        )
        # Remove invisible Unicode characters
        text = re.sub(r'[\u200B\u200C\u200D\uFEFF]', '', text)
        # Normalize Unicode representation
        return unicodedata.normalize("NFC", text)
    def normalize_smart_quotes(self, text):
        """Converts curly/smart punctuation to standard straight ASCII characters."""
        if not isinstance(text, str): return ""
        smart_map = {
            '’': "'", '‘': "'", '`': "'",
            '“': '"', '”': '"',
            '–': '-', '—': '-'  # En-dash and em-dash to regular hyphen
        }
        for curly, straight in smart_map.items():
            text = text.replace(curly, straight)
        return text
    def normalize_prices(self, text): 
        if not isinstance(text, str): return ""
        # 450k → 450000 
        text = re.sub(r'\b(\d+\.?\d*)\s*k\b', 
                      lambda m: str(int(float(m.group(1)) * 1000)), 
                      text, 
                      flags=re.I) 
        # 1.2m → 1200000 
        text = re.sub(r'\b(\d+\.?\d*)\s*m\b', 
                      lambda m: str(int(float(m.group(1))*1000000)), 
                      text, 
                      flags=re.I) 
        return text 
    def normalize_measurements(self, text):
        if not isinstance(text, str):
            return ""
        # Standardize square footage abbreviations
        text = re.sub(r'\bsq\.?\s*ft\.?\b|\bsf\b', 'sqft', text, flags=re.I)
        # Remove approximation symbols after numbers
        text = re.sub(r'(\d[\d,]*)\s*(?:±|\+/-|\+)', r'\1', text)
        # Convert dimensions like 10x12 -> 10 by 12
        text = re.sub(r'(\d+)\s*x\s*(\d+)', r'\1 by \2', text, flags=re.I)
        return text
    def normalize_html(self, text):
        """Remove HTML tags while preserving the enclosed text."""
        if not isinstance(text, str):
            return ""

        return re.sub(r"<[^>]+>", "", text)
    def _extract_top_ngrams(self, series, n=20):
        words = []
        for text in series.dropna().astype(str): 
            extracted_words = re.findall(r'\b\w+\b', text.lower())
            filtered_words = [w for w in extracted_words if w not in self.stop_words]
            words.extend(filtered_words)
        return Counter(words).most_common(n)
    def normalize_whitespace(self, text):
        if not isinstance(text, str): return ""
        text = re.sub(r'[\r\n\t]+', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    def expand_abbreviations(self, text):
        if not isinstance(text, str):
            return ""
        for abbrev, full in sorted(
            self.abbrev_map.items(),
            key=lambda x: len(x[0]),
            reverse=True
        ):
            if abbrev in ["w/", "w/o"]:
                pattern = rf"{re.escape(abbrev)}(?=\s|$)"
            else:
                pattern = rf"\b{re.escape(abbrev)}\b"
            
            def replace(match):
                original = match.group(0)
                if abbrev in self.proper_case:
                    return full
                if original[0].isupper():
                    return full.capitalize()
                return full.lower()
            text = re.sub(pattern, replace, text, flags=re.I)
        return text
    def _find_hidden_abbreviations(self, series):
        # Load all real English words from NLTK corpus
        english_words = set(w.lower() for w in nltk.corpus.words.words())

        counts = Counter()
        for text in series.dropna().astype(str):
            # Find 2-5 letter words/tokens (including things like
            # w/ or w/o if you adjust regex)
            words = re.findall(r'\b[a-zA-Z]{2,5}\b', text.lower())
            for word in words:
                # If it's not a normal English word and NOT already captured
                if word not in english_words and word not in self.abbrev_map and word not in self.words_to_ignore:
                    counts[word] += 1
            
        # Returns a list of top 20 uncaptured abbreviation
        return counts.most_common(20)
    def _detect_abbreviations(self, series):
        counts = Counter()
        for text in series.dropna():
            words = re.findall(r'\b\w+\b', text.lower())
            for word in words:
                if word in self.abbrev_map:
                    counts[word] += 1
        return counts.most_common(10)
    def _detect_unicode(self, series):
        count = 0
        for text in series.dropna():
            if any(ord(char) > 127 for char in text):
                count += 1
        return count
    def _detect_smart_quotes(self, series):
        pattern = r"[‘’“”]"
        return series.str.contains(pattern, regex=True).sum()
    def _detect_measurements(self, series):
        pattern = r'\d[\d,]*(?:[±+]|\+/-)?\s*(?:square\s+feet|sq\s*ft|sqft|\bsf\b)'
        return series.str.contains(pattern, regex=True, case=False).sum()
    def _detect_dimensions(self, series):
        pattern = r"\b\d+\s*x\s*\d+\b"
        return series.str.contains(pattern, regex=True).sum()
    def _detect_whitespace(self, series):
        pattern = r"[\r\n\t]| {2,}"
        return series.str.contains(pattern, regex=True).sum()
    def identify_phone_numbers(self, text):
        """Finds and standardizes raw phone numbers aggressively, ignoring APN formats."""
        # Clean out APNs first 
        apn_regex = r'\b\d{3,4}-\d{3}-\d{3,4}\b'
        cleaned_text = re.sub(apn_regex, '[PARCEL NUMBER]', text)
        
        # Now find and substitute actual phone layouts
        phone_regex = r'(?:\+?1[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}'
        return re.sub(phone_regex, '[PHONE NUMBER]', cleaned_text)
    # Before cleaning, understand what needs cleaning
    def profile_column(self, df, column_name): 
        """Analyze what's actually in L_Remarks""" 
        series = df[column_name]
        apn_regex = r'\b\d{3,4}-\d{3}-\d{3,4}\b'
        phone_regex = r'(?:\+?1[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}'
        parcel_count = series.str.contains(apn_regex, regex=True).sum()
        masked_series = series.str.replace(apn_regex, '[PARCEL]', regex=True)
        phone_count = masked_series.str.contains(phone_regex, regex=True).sum()
        return {
            "total_rows": len(series),
            "null_rate": series.isnull().mean(),
            "avg_length": series.str.len().mean(),
            "parcel_number_counts": parcel_count,
            "phone_number_counts": phone_count,
            "common_terms": self._extract_top_ngrams(series),
            "price_mentions":
                series.str.contains(r'\$\d').sum(),
            "has_html":
                series.str.contains('<[^>]+>').sum(),
            "measurement_mentions":
                self._detect_measurements(series),
            "room_dimensions":
                self._detect_dimensions(series),
            "unicode_issues":
                self._detect_unicode(series),
            "smart_quotes":
                self._detect_smart_quotes(series),
            "whitespace_issues":
                self._detect_whitespace(series),
            "common_abbreviations":
                self._detect_abbreviations(series),
            "unknown_abbreviations":
                self._find_hidden_abbreviations(series)
        }

In [120]:
cleaner = TextCleaner()
text = "HOA fee is $250\u200b per month"

print(cleaner.normalize_unicode(text))
print(cleaner.clean_text(text))

HOA fee is $250 per month
Homeowners Association fee is $250 per month


In [104]:
text = "Spacious master suite w/ walk-in closet"
print(cleaner.clean_text(text))

Spacious master suite with walk-in closet


In [94]:
text = "Offers welcome at <span>$450</span>k"

print(cleaner.normalize_html(text))
print(cleaner.normalize_prices(cleaner.normalize_html(text)))

Offers welcome at $450k
Offers welcome at $450000


In [105]:
text = "APN 5571-027-008 (13,570 sq ft)"
print(cleaner.clean_text(text))

Assessors Parcel Number 5571-027-008 (13,570 square feet)


In [121]:
text = "stunning 4,146± square feet"
print(cleaner.clean_text(text))

stunning 4,146 square feet


#### 1. Data Profiling (Learn the data, what is in there, what kind of noises)

In [110]:
cleaner = TextCleaner()
profile = cleaner.profile_column(df, 'remarks') 

In [111]:
print("========== REMARKS PROFILE ==========\n")

print(f"Total listings: {profile['total_rows']}")
print(f"Null rate: {profile['null_rate']:.2%}")
print(f"Average length: {profile['avg_length']:.2f} characters")

print(f"\nPrice mentions: {profile['price_mentions']}")
print(f"Measurement mentions: {profile['measurement_mentions']}")
print(f"Room dimensions: {profile['room_dimensions']}")

print(f"\nHTML tags: {profile['has_html']}")
print(f"Unicode usages: {profile['unicode_issues']}")
print(f"Smart quotes: {profile['smart_quotes']}")
print(f"Whitespace issues: {profile['whitespace_issues']}")
print(f"APN Counts: {profile['parcel_number_counts']}")
print(f"Phone Number Counts: {profile['phone_number_counts']}")

print("\nKnown abbreviations")
for word, count in profile["common_abbreviations"]:
    print(f"{word:8} {count}")

print("\nUnknown abbreviations")
for word, count in profile["unknown_abbreviations"]:
    print(f"{word:8} {count}")

print("\nTop 20 words")
for word, count in profile["common_terms"]:
    print(f"{word:12} {count}")

========== REMARKS PROFILE ==========

Total listings: 1000
Null rate: 0.00%
Average length: 1290.09 characters

Price mentions: 57
Measurement mentions: 261
Room dimensions: 7

HTML tags: 0
Unicode usages: 394
Smart quotes: 276
Whitespace issues: 482
APN Counts: 3
Phone Number Counts: 0

Known abbreviations
ft       251
sq       237
condo    139
hoa      134
adu      122
bbq      104
rv       70
ev       58
hvac     57
ac       45

Unknown abbreviations
altos    3
sony     3
ages     3
wings    3
pulls    3
hemet    3
harte    3
rents    3
sj       3
tools    3
finds    3
items    3
lofts    3
gpm      2
draws    2
verde    2
lies     2
rises    2
tells    2
baja     2

Top 20 words
home         2383
living       1964
room         1233
offers       1073
space        1068
kitchen      953
private      849
bedroom      832
s            824
dining       815
2            796
features     761
you          750
area         690
located      668
spacious     667
bedrooms     666
new          

In [112]:
print(df[df['remarks'].str.contains(r'\bapn\b', case=False, na=False)]['remarks'].iloc[0])

Perched atop a commanding promontory within a private gated enclave in the Hollywood Hills, 2000 Wattles Dr occupies a rare 18,500+ square foot parcel, offering unobstructed, cinematic views that stretch across the city skyline. The residence spans over 10,000 square feet with eight bedrooms, seven baths, and expansive terraces that frame the surrounding landscape. Walls of glass and an open architectural plan create a seamless transition between indoor and outdoor spaces, flooding the interiors with natural light while capturing sweeping views from nearly every room. The main living level unfolds as a series of elegant yet relaxed gathering spaces. An expansive family room flows into an open-concept kitchen and breakfast area, while a sunlit living room provides a dramatic vantage point over the city below. The middle level offers guest accommodations with a private entrance, three generously sized bedrooms, and a versatile open hall suited for formal dining, wellness, or fitness. On 

Exploring the different mentions

Price mentions

In [19]:
def filter_price_mentions(df, column_name):
    # The regex pattern to look for a dollar sign followed by a digit
    pattern = r'\$\d'
    
    # Return rows where the pattern is found
    return df[
        df[column_name].str.contains(
            pattern,
            regex=True,
            case=False,
            na=False
        )
    ]

# Apply the filter
price_df = filter_price_mentions(df, "remarks")
print(f"Filtered DataFrame shape: {price_df.shape}")

Filtered DataFrame shape: (57, 7)


In [25]:
def extract_and_view_prices(df, column_name):
    # Your pattern: optional $, digits, optional decimal, and 'k' or 'm'
    pattern = r"\$?\d+(?:\.\d+)?[kmKM]"

    result_df = df.copy()
    
    # Extract all matching mentions into a list per row
    result_df['extracted_prices'] = result_df[column_name].str.findall(pattern)
    
    # Optional: If you want a quick count of the most common price mentions across the whole dataset
    # We flatten the lists and grab value counts
    all_mentions = result_df['extracted_prices']
    print(all_mentions.value_counts().head(10))
    
    return all_mentions

# Run the extraction on your filtered data
price_df = extract_and_view_prices(price_df, "remarks")

extracted_prices
[]                     47
[$1M, 0.4m]             1
[$2M]                   1
[750k, 730K, $730k]     1
[$10k]                  1
[$100k, $1.7m]          1
[1.5m]                  1
[$20K]                  1
[200K, 30K]             1
[$675K]                 1
Name: count, dtype: int64


Measurement mentions

In [55]:
def filter_measurement_mentions(df, column_name):

    pattern = r"\b(sq\.?\s*ft\.?|sqft|sf)\b"

    return df[
        df[column_name].str.contains(
            pattern,
            regex=True,
            case=False,
            na=False
        )
    ]

# Apply the filter
measurement_df = filter_measurement_mentions(df, "remarks")
print(f"Filtered DataFrame shape: {measurement_df.shape}")

Filtered DataFrame shape: (217, 7)


C:\Users\mayab\AppData\Local\Temp\ipykernel_23424\3000703909.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[column_name].str.contains(


In [56]:
def extract_and_view_measurements(df, column_name):

    pattern = r"\b(sq\.?\s*ft\.?|sqft|sf)\b"

    result_df = df.copy()

    result_df["extracted_measurements"] = (
        result_df[column_name].str.findall(pattern)
    )

    all_mentions = result_df['extracted_measurements']
    print(all_mentions.value_counts().head(20))

    return all_mentions
measurement_df = extract_and_view_measurements(measurement_df, "remarks")

extracted_measurements
[sq ft]                                     67
[sq. ft]                                    44
[]                                          29
[sq ft, sq ft]                              25
[sq. ft, sq. ft]                            16
[sqft]                                      15
[sf]                                         5
[sq ft, sq ft, sq ft]                        3
[sq. ft, sq ft]                              2
[sf, sf]                                     2
[sqft, sq ft]                                2
[sq. ft, sq. ft, sq. ft]                     1
[sqft, sq ft, sqft, sqft, sqft]              1
[sq. ft, sq. ft, sq. ft, sq. ft, sq. ft]     1
[sq.ft, sq.ft]                               1
[sqft, sqft]                                 1
[sq. ft, sq.ft]                              1
[sq ft, sq ft, sq ft, sq ft]                 1
Name: count, dtype: int64


Room Dimensions

In [28]:
def filter_dimensions_mentions(df, column_name):

    pattern = r"\b\d+\s*x\s*\d+\b"

    return df[
        df[column_name].str.contains(
            pattern,
            regex=True,
            case=False,
            na=False
        )
    ]

# Apply the filter
dimensions_df = filter_dimensions_mentions(df, "remarks")
print(f"Filtered DataFrame shape: {dimensions_df.shape}")

Filtered DataFrame shape: (7, 7)


In [29]:
def extract_and_view_dimensions(df, column_name):

    pattern = r"\b\d+\s*x\s*\d+\b"

    result_df = df.copy()

    result_df["extracted_dimensions"] = (
        result_df[column_name].str.findall(pattern)
    )

    all_mentions = result_df['extracted_dimensions']
    print(all_mentions.value_counts().head(10))

    return all_mentions
dimensions_df = extract_and_view_dimensions(dimensions_df, "remarks")

extracted_dimensions
[14 x 22]         1
[12x14, 8x8]      1
[12x24]           1
[12x12]           1
[40x80, 17x28]    1
[24x24]           1
[24x48]           1
Name: count, dtype: int64


Unicode 

In [39]:
def filter_unicode_mentions(df, column_name):

    return df[
        df[column_name].apply(
            lambda x: any(ord(c) > 127 for c in str(x))
        )
    ]
# Apply the filter
unicode_df = filter_unicode_mentions(df, "remarks")
print(f"Filtered DataFrame shape: {unicode_df.shape}")

Filtered DataFrame shape: (394, 7)


In [40]:
def extract_and_view_unicode(df, column_name):
    result_df = df.copy()

    result_df["extracted_unicode"] = result_df[column_name].apply(
        lambda text: [
            c for c in str(text)
            if ord(c) > 127
        ]
    )

    all_mentions = result_df['extracted_unicode']
    print(all_mentions.value_counts().head(10))

    return all_mentions
unicode_df = extract_and_view_unicode(unicode_df, "remarks")

extracted_unicode
[’]          75
[—]          44
[’, ’]       36
[—, —]       24
[’, ’, ’]    18
[—, ’]       16
[’, —]       10
[—, —, —]     8
[’, —, ’]     7
[—, ’, —]     5
Name: count, dtype: int64


#### 2. Applying clean_text() to normalize remarks

In [122]:
df["cleaned_remarks"] = df["remarks"].apply(cleaner.clean_text)

In [123]:
df["cleaned_remarks"].isna().sum()

np.int64(0)

#### 3. Before/After: Showing Improvements (first ten examples)

In [124]:
changed_mask = df['remarks'] != df['cleaned_remarks']
df_changed = df[changed_mask]

print(f"Total rows improved: {df_changed.shape[0]}\n")

# 2. Display them side-by-side (10 examples)
print("First Ten Examples:")
for idx, row in df_changed.head(10).iterrows():
    print(f"Row ID: {idx}")
    print(f"BEFORE: {repr(row['remarks'])}")
    print(f"AFTER:  {repr(row['cleaned_remarks'])}")

Total rows improved: 866

First Ten Examples:
Row ID: 0
BEFORE: 'This unique property offers two homes on one lot, creating an exceptional opportunity for both owner-occupants and\r\r\ninvestors alike. The front home features 2 bedrooms and 1 bathroom, while the rear unit offers 1 bedroom and 1 bathroom.\r\r\nIdeal for extended family, rental income, or a live-in-one-rent-the-other setup. The owner currently lives in one unit, which\r\r\nmakes it easier to occupy or rent it out!'
AFTER:  'This unique property offers two homes on one lot, creating an exceptional opportunity for both owner-occupants and investors alike. The front home features 2 bedrooms and 1 bathroom, while the rear unit offers 1 bedroom and 1 bathroom. Ideal for extended family, rental income, or a live-in-one-rent-the-other setup. The owner currently lives in one unit, which makes it easier to occupy or rent it out!'
Row ID: 1
BEFORE: "Beautiful Two-Story Home in Victorville – Spacious, Modern, and Move-In Ready Welc

In [126]:
#df.to_csv("../data/processed/cleaned_listing_sample.csv", index=False)